### 03. IsolationForest로 같은 구역의 움직임 비교

- 같은 구역의 온도와 출력을 함께 보고 이전 LOT와 다른 조합을 찾음
  - Z4에는 `CP-Z4`도 함께 넣음
  - 구역 후보를 행 전체의 `1`로 합치지 않음
- CSV 파일명만 바꾸면 같은 세 가지 학습 방법을 실행함
  - 고정 `9/2/3`, 이전 동일 강종, 이전 모든 강종
- 기존 구역 라벨은 같은 LOT의 평균에서 3표준편차를 넘긴 후보를 바탕으로 함.
  - 검토한 공통 움직임만 후보에서 뺐고, 모든 1을 정비 기록으로 확인한 것은 아님.
  - 1분 안에 남은 후보 원기록이 하나라도 있으면 그 분의 구역 라벨은 1임.
  - 분 전체가 계속 이상이었다는 뜻이 아니며, 고장 정답으로 사용하지 않음.


### 1. 파일과 실행 옵션

- `data/processed/`에 CSV를 넣고 `DATA_FILE`만 바꿈
  - v1과 v2는 기존 행 라벨, v3, v4, v5는 구역 라벨을 보존함
  - 어떤 라벨도 IsolationForest 학습이나 후보 기준 계산에 넣지 않음
- 조업이벤트, 정비이력, 정기수리캘린더는 온도 원본과 같은 폴더에 둠
  - `data/raw/`에 온도 원본이 있으면 그 폴더를 사용함
- 기존 결과 폴더 아래 CSV 이름, 내용 식별값, 실행 시각으로 새 폴더를 만듦
- 파일이나 옵션을 바꾸면 첫 셀부터 전체 실행하고 마지막 저장 셀까지 마침.

In [ ]:
DATA_FILE = "cal01_base_v5.csv"  # 1분 집계 v5. 다른 파일도 이름만 바꿔 실행함.
RUN_FIRST_LOT = False  # True면 001의 시간 60/20/20도 실행함
RUN_NO_HISTORY_LOT = True  # 이전 동일 강종이 없으면 해당 LOT 내부 시간 분할을 별도 실행함.

import os
import json
import hashlib
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import sklearn
from matplotlib import font_manager, rcParams
from matplotlib.colors import LinearSegmentedColormap, ListedColormap
from matplotlib.patches import Patch
from IPython.display import display
from sklearn.ensemble import IsolationForest

ROOT = os.getcwd()
if not os.path.isdir(os.path.join(ROOT, "notebooks")):
    ROOT = os.path.dirname(ROOT)
if not os.path.isdir(os.path.join(ROOT, "notebooks")):
    ROOT = os.path.dirname(ROOT)

ENC = "utf-8-sig"
PROCESSED_DIR = os.path.join(ROOT, "data", "processed")
BASE_CSV = os.path.join(PROCESSED_DIR, DATA_FILE)
files = {
    "base": BASE_CSV,
    "event": "G-02_조업이벤트.csv",
    "maintenance": "G-02_정비이력.csv",
    "repair_plan": "G-01_정기수리캘린더.csv",
}
# 노션 공통 코드와 같은 폴더 선택임
DATA_DIR = os.path.join(ROOT, "data")
if os.path.isfile(os.path.join(DATA_DIR, "raw", "T-CR1-CAL01_온도.csv")):
    DATA_DIR = os.path.join(DATA_DIR, "raw")
for key in ["event", "maintenance", "repair_plan"]:
    files[key] = os.path.join(DATA_DIR, files[key])

input_hashes = {}
for key, path in files.items():
    with open(path, "rb") as source_file:
        input_hashes[key] = hashlib.sha256(source_file.read()).hexdigest()

RUN_ID = os.path.splitext(DATA_FILE)[0] + "__" + input_hashes["base"][:8] + "__" + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
OUT_DIR = os.path.join(PROCESSED_DIR, "sayletskin", "isolation_forest_ab", RUN_ID)
FIG_DIR = os.path.join(ROOT, "figures", "sayletskin", "isolation_forest_ab", RUN_ID)
os.makedirs(OUT_DIR)
os.makedirs(FIG_DIR)

font_names = []
for installed_font in font_manager.fontManager.ttflist:
    font_names.append(installed_font.name)
for font_name in ["Malgun Gothic", "AppleGothic", "NanumGothic", "NanumBarunGothic", "Noto Sans CJK KR"]:
    if font_name in font_names:
        rcParams["font.family"] = font_name
        break
rcParams["axes.unicode_minus"] = False
rcParams["figure.dpi"] = 110

SEED = 42  # 무작위 선택을 고정해 같은 입력에서 같은 결과를 얻음
N_ESTIMATORS = 60  # 나무 60개를 학습함
MAX_FIT_ROWS = 12000
MAX_SAMPLES = 256  # 나무 한 개에 넣을 최대 기록 수, 조건별 기준의 최소 행수와는 다름
MAX_GAP_SECONDS = 15
MIN_PAST_ROWS = 10
MIN_REFERENCE_ROWS = 256  # 원본 기록에서 조건별 기준을 계산할 최소 행수
MIN_REFERENCE_MINUTES = 5  # 기록이 있는 서로 다른 분이 최소 5개여야 함
MIN_MINUTE_REFERENCE_ROWS = 5  # v4는 원본 기록 256행 대신 유효한 1분 행 5개를 요구함
RAW_MIN_REFERENCE_ROWS = MIN_REFERENCE_ROWS  # 분 단위 설정으로 바꾸기 전에 원본 기록 설정을 보존함
TRAIN_QUANTILE = 0.95

TRAIN_LOTS = ["240001", "240024", "240037", "240038", "240044", "240058", "240061", "240069", "240091"]
VALID_LOTS = ["240108", "240114"]
TEST_LOTS = ["240117", "240121", "240133"]
LOTS = TRAIN_LOTS + VALID_LOTS + TEST_LOTS
groups = {
    "CLN": ["CLN-IN"],
    "Z1": ["TC-Z1", "OP-Z1"],
    "Z2": ["TC-Z2", "OP-Z2"],
    "Z3": ["TC-Z3", "OP-Z3"],
    "Z4": ["TC-Z4", "OP-Z4", "CP-Z4"],
    "Z5": ["TC-Z5", "OP-Z5"],
    "Z6": ["TC-Z6", "OP-Z6"],
    "Z7": ["TC-Z7", "OP-Z7"],
    "Z8": ["TC-Z8", "OP-Z8"],
    "Z9": ["TC-Z9"],
    "COOLING": ["TC-OUT1", "TC-OUT2"],
}

# 표에 표시할 구역이 맞는지 먼저 확인함.
if type(RUN_FIRST_LOT) is not bool:
    raise ValueError("RUN_FIRST_LOT은 True 또는 False를 따옴표 없이 입력함.")

group_titles = {
    "CLN": "세정 CLN-IN",
    "Z1": "건조 Z1",
    "Z2": "건조 Z2",
    "Z3": "가열 Z3",
    "Z4": "가열 Z4와 CP",
    "Z5": "가열 Z5",
    "Z6": "가열 Z6",
    "Z7": "과시효 Z7",
    "Z8": "과시효 Z8",
    "Z9": "과시효 Z9",
    "COOLING": "냉각 OUT1과 OUT2",
}
GRADE_GROUPS = ["Z3", "Z4", "Z5", "Z6", "Z7", "Z8", "Z9", "COOLING"]
SENSORS = []
for group_sensors in groups.values():
    SENSORS.extend(group_sensors)
META = ["ROW_ID", "LOT_NO", "MEAS_DT", "GRADE", "EXCLUDED", "EXCLUSION_REASON"]
method_names = {
    "fixed_9_2_3": "고정 9/2/3",
    "same_grade_history": "이전 동일 강종",
    "all_history": "이전 모든 강종",
    "first_lot_60_20_20": "001 시간 60/20/20",
    "within_lot_60_20_20": "같은 강종 없음: LOT 내부 60/20/20",
}
print("입력:", BASE_CSV)
print("결과:", OUT_DIR)
print("그림:", FIG_DIR)


### 2. 조업 기록과 기존 제외

- 센서가 측정된 시각과 같거나 그보다 앞선 강종변경 기록을 연결함
  - `direction="backward"`는 이 조건에서 가장 최근 기록을 찾는 옵션임
- 정비 작업 시간과 수리 계획 날짜가 겹치는지 따로 표시함
  - 정비 작업과 겹친다는 이유만으로 고장이나 정지로 확정하지 않음
- 1분 CSV는 원본 기록에 연결한 작업 기록을 그대로 읽음
  - 분 시작 시각으로 정비나 강종을 다시 연결하지 않음
  - `OBS_START`와 `OBS_END`는 실제 관측 범위, `AVAILABLE_AT`은 분의 값이 준비되는 시각임
- `058` 정기수리 전 정지, `061` 정기수리 후 재가동과 기존 제외를 유지함
  - 작업 종료 기록과 팀 판단을 사용하므로 사후 분석임

- CSV 안의 `DATA_KIND`는 원본 기록인지 1분 집계인지 나타냄.
  - 1분 집계의 `FEATURE_PROFILE`은 과거 60초 특징을 먼저 만들고 분 안에서 평균낸 방식이어야 함.
  - 필수 컬럼이 없거나 다른 계산 방식이면 실행을 멈추고 이유를 표시함.
- `Z3__LABEL`은 기존 구역 라벨이고 `Z3__FLAG`는 모델이 계산한 후보임.
  - `1`은 후보, `0`은 해당 기준에서 후보가 아니라는 뜻임.
  - 고장이나 정상을 확정한 값은 아님.
  - 라벨이 빈 평가 행도 입력과 학습 조건이 맞으면 모델 후보를 계산함.
  - 라벨이 없는 평가 행은 지표 계산에서 빼며, 모델 판정 보류와 구분함.
  - 구역 라벨에 `0`, `1`, 빈값 외의 값이 있으면 실행을 멈춤.

- v5의 `__STAT_LABEL`은 공통 움직임 해제 전의 통계 후보임.
  - `__DOMAIN_LABEL`은 별도로 남긴 검토 결과이며 미검토는 빈칸임.
  - 이번 비교 대상은 기존처럼 `__LABEL`, 즉 검토를 반영한 통계 후보임.
  - 새 라벨과 판단 이유는 모델 입력에 넣지 않음.
  - 도메인 검토 라벨만으로 학습한 실험이라고 해석하지 않음.


In [ ]:
# 결과를 다시 쓰기 전에 이전 실행의 완료 표시를 지움.
completion_file = os.path.join(OUT_DIR, "settings.json")
if os.path.isfile(completion_file):
    os.remove(completion_file)

# 분 집계 컬럼이 있는지 먼저 확인함
header = pd.read_csv(BASE_CSV, encoding=ENC, nrows=0).columns.tolist()
# 1분 집계 열이 있는데 시각 열이 빠지면 원본 기록으로 잘못 읽지 않음.
minute_suffixes = ("__VALUE", "__DIFF_PER_SEC", "__DEV_60S", "__STD_60S",
                   "__MODEL_ELIGIBLE", "__FEATURE_COUNT", "__GRADE_CHANGE_LOG_MIN")
has_minute_fields = "BIN_END" in header or "FEATURE_PROFILE" in header
has_minute_fields = has_minute_fields or any(column.endswith(minute_suffixes) for column in header)
if "BIN_START" not in header and has_minute_fields:
    raise ValueError("1분 집계 열이 있는데 BIN_START가 없습니다. CSV 생성 코드를 확인해주세요.")
DATA_KIND = "minute" if "BIN_START" in header else "raw"
CONTEXT_META = ["EVT_ID", "EVT_DT", "BEF_VAL", "AFT_VAL", "MINUTES_SINCE_GRADE_CHANGE",
                "EVENT_MATCH", "MAINT_OVERLAP", "MAINT_WORK_IDS", "REPAIR_PLAN_IDS",
                "CONTEXT", "OPERATING_NOTE"]
TIME_META = ["OBS_START", "OBS_END", "AVAILABLE_AT"]
LABEL_COLUMNS = []
LABEL_META = []
for group in groups:
    column = group + "__LABEL"
    if column in header:
        LABEL_COLUMNS.append(column)
    for suffix in ["LABEL", "LABEL_STATUS", "LABEL_REASON", "LABEL_SENSORS", "COMMON_REVIEW_IDS",
                   "LABEL_1_COUNT", "LABEL_0_COUNT", "LABEL_UNKNOWN_COUNT", "LABEL_1_RATIO",
                   "LABEL_FIRST_1_DT", "LABEL_LAST_1_DT"]:
        column = group + "__" + suffix
        if column in header:
            LABEL_META.append(column)

# v1과 v2의 행 라벨, v3의 이전 라벨은 이름을 바꾸지 않고 참고용으로 남김
for column in ["LABEL", "LABEL_V1", "DATA_KIND", "DATASET_VERSION", "FEATURE_PROFILE"]:
    if column in header:
        META.append(column)
META = META + LABEL_META
read_columns = META + SENSORS
minute_feature_columns = []
MINUTE_META = []
if DATA_KIND == "minute":
    MINUTE_META = ["BIN_START", "BIN_END", "ROW_COUNT", "OBSERVED_SPAN_SECONDS",
                   "SOURCE_FIRST_ROW_ID", "SOURCE_LAST_ROW_ID"]
    for sensor in SENSORS:
        for feature in ["VALUE", "DIFF_PER_SEC", "DEV_60S", "STD_60S"]:
            minute_feature_columns.append(sensor + "__" + feature)
    for group in groups:
        minute_feature_columns.append(group + "__MODEL_ELIGIBLE")
        minute_feature_columns.append(group + "__FEATURE_COUNT")
        MINUTE_META.append(group + "__MODEL_STATUS")
        if group in GRADE_GROUPS:
            minute_feature_columns.append(group + "__GRADE_CHANGE_LOG_MIN")
    missing_features = [column for column in minute_feature_columns if column not in header]
    if missing_features:
        raise ValueError("1분 입력에 필요한 파생 열이 없습니다: " + ", ".join(missing_features))
    read_columns = read_columns + CONTEXT_META + TIME_META + ["SEGMENT"] + MINUTE_META + minute_feature_columns

available_columns = []
for column in read_columns:
    if column in header and column not in available_columns:
        available_columns.append(column)

df = pd.read_csv(
    BASE_CSV, encoding=ENC, usecols=available_columns,
    dtype={"LOT_NO": "string", "GRADE": "string"}, low_memory=False,
)
# 파일명은 자유롭게 바꿔도 되며, 계산 방식은 파일 안의 표시로 확인함.
if "DATA_KIND" in df:
    declared_kind = df["DATA_KIND"].astype("string").str.strip()
    if not declared_kind.eq(DATA_KIND).fillna(False).all():
        raise ValueError("DATA_KIND와 시간 열 구성이 다릅니다. 원본과 1분 집계를 섞지 마세요.")
elif DATA_KIND == "minute":
    raise ValueError("1분 CSV에는 DATA_KIND=minute이 필요합니다.")

if DATA_KIND == "minute":
    if "FEATURE_PROFILE" not in df:
        raise ValueError("1분 CSV에는 FEATURE_PROFILE이 필요합니다.")
    profile = df["FEATURE_PROFILE"].astype("string").str.strip()
    if not profile.eq("raw60s_groupmean_1min").fillna(False).all():
        raise ValueError("1분 CSV의 모든 FEATURE_PROFILE은 raw60s_groupmean_1min이어야 합니다.")

df["MEAS_DT"] = pd.to_datetime(df["MEAS_DT"])
df["EXCLUDED"] = ~df["EXCLUDED"].astype("string").str.strip().str.lower().isin(["false", "0", "0.0"])
df["EXCLUSION_REASON"] = df["EXCLUSION_REASON"].fillna("")
df["GRADE"] = df["GRADE"].fillna("미기록")

for column in LABEL_COLUMNS:
    label_values = pd.to_numeric(df[column], errors="coerce")
    has_label = df[column].astype("string").str.strip().fillna("").ne("")
    if (has_label & ~label_values.isin([0, 1])).any():
        raise ValueError(column + "에는 0, 1, 빈값만 넣어주세요.")
    df[column] = label_values.where(label_values.isin([0, 1]))

for column in ["LABEL", "LABEL_V1"]:
    if column in df:
        df[column] = pd.to_numeric(df[column], errors="coerce")

for sensor in SENSORS:
    df[sensor] = pd.to_numeric(df[sensor], errors="coerce").replace([np.inf, -np.inf], np.nan)

if DATA_KIND == "raw":
    events = pd.read_csv(files["event"], encoding=ENC)
    events = events.loc[events["PLANT_CD"].eq("CR1") & events["EVT_TYPE"].eq("강종변경")].copy()
    events["EVT_DT"] = pd.to_datetime(events["EVT_DT"])
    events = events.sort_values("EVT_DT")
    # backward는 관측 시각보다 앞서거나 같은 시각의 가장 최근 강종변경을 연결함
    # 미래 조업이벤트를 끌어오지 않음
    df = pd.merge_asof(
        df.sort_values("MEAS_DT"),
        events[["EVT_ID", "EVT_DT", "BEF_VAL", "AFT_VAL"]],
        left_on="MEAS_DT", right_on="EVT_DT", direction="backward",
    )
    df = df.sort_values(["LOT_NO", "MEAS_DT"], kind="stable").reset_index(drop=True)
    df["MINUTES_SINCE_GRADE_CHANGE"] = (df["MEAS_DT"] - df["EVT_DT"]).dt.total_seconds() / 60
    df["EVENT_MATCH"] = df["GRADE"].eq(df["AFT_VAL"]).fillna(False) & df["EVT_DT"].notna()

    maintenance = pd.read_csv(files["maintenance"], encoding=ENC)
    maintenance = maintenance.loc[maintenance["EQP_TAG"].eq("P1-CR1-CAL01")].copy()
    for column in ["STRT_DT", "END_DT"]:
        maintenance[column] = pd.to_datetime(maintenance[column])
    df["MAINT_WORK_IDS"] = ""
    for work in maintenance.itertuples(index=False):
        overlap = df["MEAS_DT"].ge(work.STRT_DT) & df["MEAS_DT"].lt(work.END_DT)
        df.loc[overlap, "MAINT_WORK_IDS"] += work.WO_NO + "|"
    df["MAINT_WORK_IDS"] = df["MAINT_WORK_IDS"].str.rstrip("|")
    df["MAINT_OVERLAP"] = df["MAINT_WORK_IDS"].ne("")

    repair = pd.read_csv(files["repair_plan"], encoding=ENC)
    repair = repair.loc[repair["PLANT_CD"].eq("CR1")].copy()
    repair["STRT_DT"] = pd.to_datetime(repair["STRT_DT"])
    repair["END_EXCLUSIVE"] = pd.to_datetime(repair["END_DT"]).dt.normalize() + pd.Timedelta(days=1)
    df["REPAIR_PLAN_IDS"] = ""
    for plan in repair.itertuples(index=False):
        overlap = df["MEAS_DT"].ge(plan.STRT_DT) & df["MEAS_DT"].lt(plan.END_EXCLUSIVE)
        df.loc[overlap, "REPAIR_PLAN_IDS"] += plan.PLAN_ID + "|"
    df["REPAIR_PLAN_IDS"] = df["REPAIR_PLAN_IDS"].str.rstrip("|")

    df["CONTEXT"] = "작업 기록 없음"
    df.loc[df["MAINT_OVERLAP"], "CONTEXT"] = "정비 작업 기록과 겹침"
    df.loc[df["REPAIR_PLAN_IDS"].ne(""), "CONTEXT"] = "정기수리 계획 기간"
    df["OPERATING_NOTE"] = "별도 운전 상태 기록 없음"
    df.loc[df["EXCLUDED"], "OPERATING_NOTE"] = df.loc[df["EXCLUDED"], "EXCLUSION_REASON"]
    df.loc[df["EXCLUDED"] & df["LOT_NO"].eq("240058"), "OPERATING_NOTE"] = "정기수리를 위해 온도를 내림, 팀 확정"
    df.loc[df["EXCLUDED"] & df["LOT_NO"].eq("240061"), "OPERATING_NOTE"] = "정기수리 후 재가동하며 온도를 올림, 팀 확정"


    # 원본 기록 한 행은 해당 시각에 사용할 수 있음
    df["OBS_START"] = df["MEAS_DT"]
    df["OBS_END"] = df["MEAS_DT"]
    df["AVAILABLE_AT"] = df["MEAS_DT"]
else:
    # v4는 원본 기록 때 연결한 강종과 작업 기록을 유지함
    # 분 끝 시각으로 다시 연결하면 다음 강종이 붙을 수 있어 다시 연결하지 않음
    for column in TIME_META + ["BIN_START", "BIN_END", "EVT_DT"]:
        df[column] = pd.to_datetime(df[column])
    for column in ["EVENT_MATCH", "MAINT_OVERLAP"]:
        df[column] = df[column].astype("string").str.strip().str.lower().isin(["true", "1", "1.0"])
    for column in ["MAINT_WORK_IDS", "REPAIR_PLAN_IDS", "OPERATING_NOTE"]:
        df[column] = df[column].fillna("")
    for column in minute_feature_columns:
        if column.endswith("__MODEL_ELIGIBLE"):
            df[column] = df[column].astype("string").str.strip().str.lower().isin(["true", "1", "1.0"])
        else:
            df[column] = pd.to_numeric(df[column], errors="coerce").replace([np.inf, -np.inf], np.nan)
    df = df.sort_values(["LOT_NO", "MEAS_DT", "SEGMENT"], kind="stable").reset_index(drop=True)
    MIN_REFERENCE_ROWS = MIN_MINUTE_REFERENCE_ROWS

# 판정 결과에 라벨과 실제 관측 범위를 함께 남김
for column in MINUTE_META:
    if column in df and column not in META:
        META.append(column)
for column in minute_feature_columns:
    if column.endswith("__MODEL_ELIGIBLE") or column.endswith("__FEATURE_COUNT"):
        META.append(column)

DATASET_VERSION = "파일에 버전 기록 없음"
if "DATASET_VERSION" in df:
    DATASET_VERSION = str(df["DATASET_VERSION"].iloc[0])
source_manifest = {}
manifest_status = "해당 버전의 metadata 없음"
manifest_dir = os.path.join(PROCESSED_DIR, "metadata", DATASET_VERSION)
manifest_path = os.path.join(manifest_dir, input_hashes["base"] + ".json")
if not os.path.isfile(manifest_path):
    manifest_path = os.path.join(manifest_dir, "dataset.json")
if os.path.isfile(manifest_path):
    with open(manifest_path, encoding="utf-8") as manifest_file:
        checked_manifest = json.load(manifest_file)
    with open(manifest_path, "rb") as manifest_file:
        input_hashes["metadata"] = hashlib.sha256(manifest_file.read()).hexdigest()
    if checked_manifest.get("master_sha256") == input_hashes["base"]:
        source_manifest = checked_manifest
        manifest_status = "선택 CSV와 내용이 일치하는 metadata를 실행 설정에 보존함"
    else:
        manifest_status = "선택 CSV와 metadata의 내용 식별값이 다름. 생성 근거는 연결하지 않음"

print("입력 단위:", DATA_KIND, "| 버전:", DATASET_VERSION)
print("CSV 생성 기록:", manifest_status)
print("조건별 최소 행수:", MIN_REFERENCE_ROWS, "| 최소 관측 분:", MIN_REFERENCE_MINUTES)

context_summary = df.groupby(["LOT_NO", "GRADE", "CONTEXT", "OPERATING_NOTE"], dropna=False).agg(
    ROWS=("ROW_ID", "size"), START=("OBS_START", "min"), END=("OBS_END", "max"),
).reset_index()
context_summary.to_csv(os.path.join(OUT_DIR, "operating_context.csv"), index=False, encoding=ENC)
display(context_summary.rename(columns={
    "LOT_NO": "LOT", "GRADE": "강종", "CONTEXT": "작업 기록과의 관계",
    "OPERATING_NOTE": "팀 판단 또는 상태 기록", "ROWS": "행수", "START": "첫 시각", "END": "마지막 시각",
}))


### 3. 같은 구역의 온도(TC)와 출력(OP)를 함께 입력

| 입력 | 의미 |
| --- | --- |
| `VALUE` | 현재 센서값 |
| `DIFF_PER_SEC` | 직전 값과의 차이를 실제 지난 초로 나눈 값 |
| `DEV_60S` | 현재값에서 직전 60초 평균을 뺀 값 |
| `STD_60S` | 직전 60초 값이 흔들린 정도 |

- 원본 기록은 현재값을 뺀 과거 60초로 계산함
  - 직전 60초 안에 유효한 기록이 10개 이상이면 계산함.
  - 60초 전체가 관측됐다는 뜻은 아님.
  - LOT, 15초 초과 공백, 제외 여부, 강종, 작업 조건이 바뀌면 과거 계산을 끊음
- 1분 CSV는 원본 기록에서 계산한 위 특징의 1분 평균을 읽음
  - 1분 자료에 15초 공백 기준이나 60초 이동 계산을 다시 적용하지 않음
  - `구역__MODEL_ELIGIBLE=False`인 분은 그 구역을 보류함
- 세정과 건조는 강종을 나누지 않음
  - 나머지 구역은 강종별 기준과 실제 강종변경 후 지난 시간도 사용함
- 라벨 근거와 중앙 60% 온도 특징은 모델 입력에서 뺌
  - TC와 OP가 반대로 움직이는 것 자체를 이상 규칙으로 넣지 않음

In [ ]:
if DATA_KIND == "raw":
    seconds = df["MEAS_DT"].diff().dt.total_seconds()
    breaks = df["LOT_NO"].ne(df["LOT_NO"].shift()).fillna(True)
    breaks = breaks | seconds.gt(MAX_GAP_SECONDS) | seconds.le(0)
    breaks = breaks | df["EXCLUDED"].ne(df["EXCLUDED"].shift())
    breaks = breaks | df["GRADE"].ne(df["GRADE"].shift())
    breaks = breaks | df["EVT_ID"].ne(df["EVT_ID"].shift()).fillna(True)
    breaks = breaks | df["CONTEXT"].ne(df["CONTEXT"].shift())
    df["SEGMENT"] = breaks.cumsum()
    features = {}

    for sensor in SENSORS:
        feature = pd.DataFrame(np.nan, index=df.index, columns=["VALUE", "DIFF_PER_SEC", "DEV_60S", "STD_60S"])
        for segment, block in df.loc[~df["EXCLUDED"]].groupby("SEGMENT", sort=False):
            values = pd.Series(block[sensor].to_numpy(), index=pd.DatetimeIndex(block["MEAS_DT"]))
            elapsed = values.index.to_series().diff().dt.total_seconds()
            past = values.rolling("60s", closed="left", min_periods=MIN_PAST_ROWS)
            feature.loc[block.index, "VALUE"] = values.to_numpy()
            feature.loc[block.index, "DIFF_PER_SEC"] = values.diff().div(elapsed).to_numpy()
            feature.loc[block.index, "DEV_60S"] = values.sub(past.mean()).to_numpy()
            feature.loc[block.index, "STD_60S"] = past.std().to_numpy()
        features[sensor] = feature.replace([np.inf, -np.inf], np.nan).astype("float32")

    group_inputs = {}
    group_valid = {}
    for group, sensors in groups.items():
        parts = []
        for sensor in sensors:
            parts.append(features[sensor].add_prefix(sensor + "__"))
        group_input = pd.concat(parts, axis=1)
        if group in GRADE_GROUPS:
            group_input["GRADE_CHANGE_LOG_MIN"] = np.log1p(df["MINUTES_SINCE_GRADE_CHANGE"])
        group_inputs[group] = group_input.astype("float32")
        valid = np.isfinite(group_input.to_numpy()).all(axis=1) & ~df["EXCLUDED"].to_numpy()
        if group in GRADE_GROUPS:
            valid = valid & df["EVENT_MATCH"].to_numpy(dtype=bool)
        group_valid[group] = valid

else:
    # v4의 특징은 원본 기록에서 60초 계산을 마친 뒤 1분씩 평균 낸 값임
    # 1분 간격 자료에 60초 rolling을 다시 적용하지 않음
    group_inputs = {}
    group_valid = {}
    for group, sensors in groups.items():
        columns = []
        for sensor in sensors:
            for feature in ["VALUE", "DIFF_PER_SEC", "DEV_60S", "STD_60S"]:
                columns.append(sensor + "__" + feature)
        group_input = df[columns].copy()
        if group in GRADE_GROUPS:
            group_input["GRADE_CHANGE_LOG_MIN"] = df[group + "__GRADE_CHANGE_LOG_MIN"]
        group_inputs[group] = group_input.astype("float32")
        valid = np.isfinite(group_input.to_numpy()).all(axis=1)
        valid = valid & df[group + "__MODEL_ELIGIBLE"].to_numpy() & ~df["EXCLUDED"].to_numpy()
        if group in GRADE_GROUPS:
            valid = valid & df["EVENT_MATCH"].to_numpy(dtype=bool)
        group_valid[group] = valid

# 관측 범위와 값을 사용할 수 있는 시각을 따로 보존함
lot_info = df.groupby("LOT_NO", sort=False).agg(
    GRADE=("GRADE", "first"), START=("OBS_START", "min"),
    END=("OBS_END", "max"), AVAILABLE_END=("AVAILABLE_AT", "max"),
    TOTAL_ROWS=("ROW_ID", "size"),
)
group_table = []
for group, sensors in groups.items():
    group_table.append({
        "구역": group_titles[group], "함께 입력하는 센서": ", ".join(sensors),
        "강종 기준": "같은 강종" if group in GRADE_GROUPS else "강종 구분 없음",
        "계산 가능한 행": int(group_valid[group].sum()),
        "구역 라벨": group + "__LABEL" if group + "__LABEL" in df else "기존 행 라벨만 있음",
    })
display(pd.DataFrame(group_table))


### 4. 이전 LOT로 학습하고 다음 LOT 확인

- 고정 `9/2/3`은 앞 9개 학습, `108`과 `114` 검증, `117`과 `121`과 `133` 시험임
- 이전 동일 강종과 이전 모든 강종은 대상 LOT보다 먼저 사용 가능해진 기록만 학습함
- `RUN_FIRST_LOT=True`면 `001`의 실제 관측 시간으로 `60/20/20`도 실행함
  - `True`와 `False`는 따옴표 없이 적음
  - 경계는 `OBS_START`의 처음부터 `OBS_END`의 마지막까지로 계산함
  - 1분 CSV에서 경계를 걸치거나 경계 전에 값이 준비되지 않은 분은 앞쪽 학습에 넣지 않음
- Valid는 결과 확인용이며 Test에 맞춰 기준을 바꾸지 않음

- `RUN_NO_HISTORY_LOT=True`이면 앞선 동일 강종이 없는 LOT도 따로 실행함.
  - 해당 LOT의 실제 시간 앞 60%를 Train, 다음 20%를 Valid, 뒤 20%를 Test로 둠.
  - 이름은 `같은 강종 없음: LOT 내부 60/20/20`임.
  - 고정 9/2/3과 학습 자료가 다르므로 공통 Test 비교에 섞지 않음.
- 내부 분할에서도 학습하지 못한 작업 조건은 보류함.
  - 분할했다고 모든 구간을 반드시 예측할 수 있는 것은 아님.

In [ ]:
# 결과를 다시 쓰기 전에 이전 실행의 완료 표시를 지움.
completion_file = os.path.join(OUT_DIR, "settings.json")
if os.path.isfile(completion_file):
    os.remove(completion_file)

jobs = [{"case": "fixed_9_2_3", "method": "fixed_9_2_3", "roles": {"train": TRAIN_LOTS, "valid": VALID_LOTS, "test": TEST_LOTS}}]

for method in ["same_grade_history", "all_history"]:
    for lot in LOTS:
        history = []
        if lot in lot_info.index:
            previous = lot_info.loc[lot_info["AVAILABLE_END"].lt(lot_info.loc[lot, "START"])].sort_values("START")
            if method == "same_grade_history":
                previous = previous.loc[previous["GRADE"].eq(lot_info.loc[lot, "GRADE"])]
            history = previous.index.tolist()
        jobs.append({"case": method + "_" + lot, "method": method, "roles": {"train": history, "test": [lot]}})

if RUN_FIRST_LOT and not RUN_NO_HISTORY_LOT and "240001" in lot_info.index:
    first_start = lot_info.loc["240001", "START"]
    first_span = lot_info.loc["240001", "END"] - first_start
    jobs.append({
        "case": "first_lot_60_20_20",
        "method": "first_lot_60_20_20",
        "roles": {"train": ["240001"], "valid": ["240001"], "test": ["240001"]},
        "cuts": [first_start + first_span * 0.6, first_start + first_span * 0.8],
    })


# 동일 강종 기록이 없는 LOT도 자기 LOT의 앞부분으로 학습하는 실험을 따로 남김.
# 고정 9/2/3 Test를 학습에 섞지 않고, 독립된 CASE로 실행함.
if RUN_NO_HISTORY_LOT:
    no_history_lots = []
    for job in jobs:
        if job["method"] == "same_grade_history" and not job["roles"]["train"]:
            no_history_lots.extend(job["roles"]["test"])
    for lot in no_history_lots:
        if lot not in lot_info.index:
            continue
        start = lot_info.loc[lot, "START"]
        span = lot_info.loc[lot, "END"] - start
        jobs.append({
            "case": "within_lot_60_20_20_" + lot,
            "method": "within_lot_60_20_20",
            "roles": {"train": [lot], "valid": [lot], "test": [lot]},
            "cuts": [start + span * 0.6, start + span * 0.8],
        })

plan_rows = []
for job in jobs:
    job["indices"] = {}
    for role, lots in job["roles"].items():
        selected = df["LOT_NO"].isin(lots)
        if "cuts" in job:
            if role == "train":
                selected = selected & df["AVAILABLE_AT"].lt(job["cuts"][0])
            elif role == "valid":
                selected = selected & df["OBS_START"].ge(job["cuts"][0]) & df["AVAILABLE_AT"].lt(job["cuts"][1])
            else:
                selected = selected & df["OBS_START"].ge(job["cuts"][1])
        job["indices"][role] = df.index[selected].to_numpy()

    # 분 집계가 평가 시작 뒤에 완성되는 경우는 학습에 넣지 않음
    evaluation_starts = []
    for role in ["valid", "test"]:
        if role in job["indices"] and len(job["indices"][role]):
            evaluation_starts.append(df.loc[job["indices"][role], "OBS_START"].min())
    if evaluation_starts:
        train_index = job["indices"]["train"]
        ready = df.loc[train_index, "AVAILABLE_AT"].lt(min(evaluation_starts)).to_numpy()
        job["indices"]["train"] = train_index[ready]

    train_index = job["indices"]["train"]
    job["train_index"] = train_index[~df.loc[train_index, "EXCLUDED"].to_numpy()]
    job["status"] = "실행" if len(job["train_index"]) else "생략"
    job["reason"] = ""
    if not job["roles"]["train"]:
        job["reason"] = "이전 동일 강종 기록 없음" if job["method"] == "same_grade_history" else "이전 LOT 기록 없음"
    elif not len(job["train_index"]):
        job["reason"] = "학습에 사용할 제외되지 않은 행 없음"
    if job["method"] == "fixed_9_2_3":
        missing_lots = []
        for required_lot in LOTS:
            if required_lot not in lot_info.index:
                missing_lots.append(required_lot)
        if missing_lots:
            job["train_index"] = np.array([], dtype=int)
            job["status"] = "생략"
            job["reason"] = "고정 9/2/3에 필요한 LOT 없음: " + ",".join(missing_lots)

    for role, lots in job["roles"].items():
        for lot in lots:
            index = job["indices"][role]
            index = index[df.loc[index, "LOT_NO"].eq(lot).to_numpy(dtype=bool)]
            n_common = int((~df.loc[index, "EXCLUDED"]).sum())
            status, reason = job["status"], job["reason"]
            if not len(index):
                status, reason = "생략", "CSV에 해당 LOT 또는 시간 구간 없음"
            elif not n_common:
                status, reason = "생략", "계산에 사용할 제외되지 않은 행 없음"
            plan_rows.append({
                "CASE": job["case"],
                "METHOD": job["method"],
                "ROLE": role,
                "LOT_NO": lot,
                "GRADE": lot_info.loc[lot, "GRADE"] if lot in lot_info.index else "",
                "START": df.loc[index, "OBS_START"].min(),
                "END": df.loc[index, "OBS_END"].max(),
                "AVAILABLE_END": df.loc[index, "AVAILABLE_AT"].max(),
                "STATUS": status,
                "REASON": reason,
                "TOTAL_ROWS": len(index),
                "COMMON_ROWS": n_common,
                "TRAIN_LOTS": ",".join(job["roles"]["train"]),
            })

split_plan = pd.DataFrame(plan_rows)
split_plan.to_csv(os.path.join(OUT_DIR, "split_plan.csv"), index=False, encoding=ENC)
display(split_plan.loc[split_plan["ROLE"].ne("train"), ["METHOD", "ROLE", "LOT_NO", "TRAIN_LOTS", "COMMON_ROWS", "STATUS", "REASON"]].rename(columns={
    "METHOD": "방법",
    "ROLE": "역할",
    "LOT_NO": "LOT",
    "TRAIN_LOTS": "학습 LOT",
    "COMMON_ROWS": "기존 제외를 뺀 행",
    "STATUS": "상태",
    "REASON": "이유",
}))


### 5. 같은 조건의 Train 점수와 비교

- 정비 작업 조건별로 모델을 나눔
  - 강종에 반응하는 구역은 Train의 강종별 중앙값과 가운데 50% 폭으로 입력 수준을 맞춤
  - 폭이 0이면 표준편차를 쓰고 그것도 0이면 1로 나눔
- `MAX_FIT_ROWS=12000`은 구역과 작업 조건별 학습 기록의 상한임.
  - 각 LOT에 같은 최대 수를 배정하며 짧은 LOT은 있는 기록만 사용함.
- 후보 기준은 뽑은 표본이 아닌 같은 조건의 Train 전체 유효 기록에서 계산함.
  - `TRAIN_QUANTILE=0.95`는 점수를 작은 순서로 놓았을 때 `95%` 위치의 값임.
  - 같은 강종과 작업 조건에서 이 값을 넘으면 구역 후보로 표시함
  - 정상 범위로 확정한 값은 아님
- 원본 기록은 조건별 `256행`과 기록이 있는 `5개 분`이 필요함
  - 1분 CSV는 `5개 유효한 1분 행`과 기록이 있는 `5개 분`이 필요함
  - 계산용 최소 설정이며 충분한 정상 자료라는 뜻은 아님
- 같은 조건의 과거 기준이 없으면 판정을 비워 두고 이유를 남김
  - 강종변경 후 지난 시간이 달라서 후보로 잡힐 수도 있음
  - 강종별 수준을 맞춘 뒤 함께 학습하며 강종별 관계 모델을 따로 만든 것은 아님

- `Z3__SCORE` 같은 점수는 클수록 모델이 드문 조합으로 본 값임
  - 고장 확률이 아니며 후보 기준을 넘었는지와 함께 읽음

In [ ]:
# 결과를 다시 쓰기 전에 이전 실행의 완료 표시를 지움.
completion_file = os.path.join(OUT_DIR, "settings.json")
if os.path.isfile(completion_file):
    os.remove(completion_file)

threshold_rows = []
reference_rows = []
fit_parts = []
prediction_parts = []

for job in jobs:
    evaluation_index = np.array([], dtype=int)
    role_values = []
    for role in ["valid", "test"]:
        if role in job["indices"]:
            index = job["indices"][role]
            evaluation_index = np.concatenate([evaluation_index, index])
            role_values.extend([role] * len(index))

    result = df.loc[evaluation_index, META + CONTEXT_META + TIME_META + ["SEGMENT"]].copy()
    result["CASE"] = job["case"]
    result["METHOD"] = job["method"]
    result["ROLE"] = role_values
    train_index = job["train_index"]

    for group, group_input in group_inputs.items():
        score_col = group + "__SCORE"
        flag_col = group + "__FLAG"
        status_col = group + "__STATUS"
        result[score_col] = np.nan
        result[flag_col] = np.nan
        result[status_col] = "같은 조업 맥락 없음"
        result.loc[~group_valid[group][evaluation_index], status_col] = "입력 부족"
        if group in GRADE_GROUPS:
            result.loc[~result["EVENT_MATCH"], status_col] = "강종 연결 불일치"
        result.loc[result["EXCLUDED"], status_col] = "기존 제외"

        if not len(train_index):
            mask = ~result["EXCLUDED"] & group_valid[group][evaluation_index]
            result.loc[mask, status_col] = job["reason"] or "이전 LOT 없음"
            continue

        usable_train = train_index[group_valid[group][train_index]]
        usable_eval = evaluation_index[group_valid[group][evaluation_index]]
        grade_key = df["GRADE"] if group in GRADE_GROUPS else pd.Series("강종 구분 없음", index=df.index)

        for context in df.loc[usable_train, "CONTEXT"].unique():
            context_train = usable_train[df.loc[usable_train, "CONTEXT"].eq(context).to_numpy()]
            context_eval = usable_eval[df.loc[usable_eval, "CONTEXT"].eq(context).to_numpy()]
            if not len(context_eval):
                continue
            result.loc[context_eval, status_col] = "같은 강종 기준 없음"
            normalized_train = pd.DataFrame(np.nan, index=context_train, columns=group_input.columns)
            normalized_eval = pd.DataFrame(np.nan, index=context_eval, columns=group_input.columns)
            references = []

            for grade in grade_key.loc[context_train].unique():
                reference_index = context_train[grade_key.loc[context_train].eq(grade).to_numpy()]
                target_index = context_eval[grade_key.loc[context_eval].eq(grade).to_numpy()]
                minute_count = df.loc[reference_index, "MEAS_DT"].dt.floor("min").nunique()
                lot_count = df.loc[reference_index, "LOT_NO"].nunique()
                enough = len(reference_index) >= MIN_REFERENCE_ROWS and minute_count >= MIN_REFERENCE_MINUTES
                if not enough:
                    result.loc[target_index, status_col] = "기준 표본 부족"
                    reference_rows.append({
                        "CASE": job["case"], "GROUP": group, "CONTEXT": context, "GRADE_KEY": grade,
                        "TRAIN_ROWS": len(reference_index), "OBSERVED_MINUTES": minute_count,
                        "LOT_COUNT": lot_count, "FEATURE": "", "CENTER": np.nan,
                        "IQR": np.nan, "SCALE": np.nan, "STATUS": "기준 표본 부족",
                    })
                    continue

                values = group_input.loc[reference_index]
                center = values.median()
                iqr = values.quantile(0.75) - values.quantile(0.25)
                scale = iqr.where(iqr.gt(0), values.std(ddof=0))
                scale = scale.where(scale.gt(0), 1.0)
                normalized_train.loc[reference_index] = (values - center) / scale
                normalized_eval.loc[target_index] = (group_input.loc[target_index] - center) / scale
                references.append((grade, reference_index, target_index, minute_count, lot_count))
                for feature_name in group_input.columns:
                    reference_rows.append({
                        "CASE": job["case"], "GROUP": group, "CONTEXT": context, "GRADE_KEY": grade,
                        "TRAIN_ROWS": len(reference_index), "OBSERVED_MINUTES": minute_count,
                        "LOT_COUNT": lot_count, "FEATURE": feature_name,
                        "CENTER": float(center[feature_name]), "IQR": float(iqr[feature_name]),
                        "SCALE": float(scale[feature_name]), "STATUS": "기준 계산",
                    })

            normalized_train = normalized_train.dropna().astype("float32")
            if normalized_train.empty:
                continue
            fit_candidates = normalized_train.index.to_numpy()
            train_lots = df.loc[fit_candidates, "LOT_NO"].unique()
            per_lot = MAX_FIT_ROWS // len(train_lots)
            picked = []
            rng = np.random.default_rng(SEED)
            for lot in train_lots:
                index = fit_candidates[df.loc[fit_candidates, "LOT_NO"].eq(lot).to_numpy(dtype=bool)]
                picked.extend(rng.choice(index, size=min(per_lot, len(index)), replace=False).tolist())
            fit_index = np.sort(np.array(picked, dtype=int))
            fit_record = df.loc[fit_index, ["ROW_ID", "LOT_NO", "MEAS_DT", "OBS_START", "OBS_END", "AVAILABLE_AT", "GRADE", "CONTEXT"]].copy()
            fit_record["CASE"] = job["case"]
            fit_record["GROUP"] = group
            fit_parts.append(fit_record)

            # 모델 기본 판정선 대신 아래의 Train 95% 기준으로 후보를 정함.
            model = IsolationForest(
                n_estimators=N_ESTIMATORS,
                max_samples=min(MAX_SAMPLES, len(fit_index)),
                contamination="auto",  # 기본 판정선 설정이며 여기서는 사용하지 않음
                random_state=SEED,  # 나무를 만들 때의 무작위 선택을 고정함
                n_jobs=2,  # 나무 학습을 동시에 두 작업으로 나눠 실행함
            )
            model.fit(normalized_train.loc[fit_index].to_numpy())
            train_scores = pd.Series(-model.score_samples(normalized_train.to_numpy()), index=normalized_train.index)
            for grade, reference_index, target_index, minute_count, lot_count in references:
                threshold = float(train_scores.loc[reference_index].quantile(TRAIN_QUANTILE))
                threshold_rows.append({
                    "CASE": job["case"], "METHOD": job["method"], "GROUP": group,
                    "CONTEXT": context, "GRADE_KEY": grade, "THRESHOLD": threshold,
                    "TRAIN_ROWS": len(reference_index), "OBSERVED_MINUTES": minute_count,
                    "LOT_COUNT": lot_count, "FIT_ROWS": len(fit_index),
                    "TRAIN_LOTS": ",".join(df.loc[reference_index, "LOT_NO"].unique()),
                    "MODEL_TRAIN_LOTS": ",".join(train_lots),
                })
                if len(target_index):
                    score = -model.score_samples(normalized_eval.loc[target_index].to_numpy(dtype="float32"))
                    result.loc[target_index, score_col] = score
                    result.loc[target_index, flag_col] = (score > threshold).astype("int8")
                    result.loc[target_index, status_col] = "계산 완료"

    prediction_parts.append(result)
    print(job["case"], "| 평가 기록", len(result))

predictions = pd.concat(prediction_parts, ignore_index=True)
thresholds = pd.DataFrame(threshold_rows, columns=[
    "CASE", "METHOD", "GROUP", "CONTEXT", "GRADE_KEY", "THRESHOLD", "TRAIN_ROWS",
    "OBSERVED_MINUTES", "LOT_COUNT", "FIT_ROWS", "TRAIN_LOTS", "MODEL_TRAIN_LOTS",
])
references = pd.DataFrame(reference_rows, columns=[
    "CASE", "GROUP", "CONTEXT", "GRADE_KEY", "TRAIN_ROWS", "OBSERVED_MINUTES",
    "LOT_COUNT", "FEATURE", "CENTER", "IQR", "SCALE", "STATUS",
])
fit_rows = pd.concat(fit_parts, ignore_index=True) if fit_parts else pd.DataFrame(columns=[
    "ROW_ID", "LOT_NO", "MEAS_DT", "OBS_START", "OBS_END", "AVAILABLE_AT", "GRADE", "CONTEXT", "CASE", "GROUP",
])
predictions.to_csv(os.path.join(OUT_DIR, "predictions.csv"), index=False, encoding=ENC, float_format="%.17g")
thresholds.to_csv(os.path.join(OUT_DIR, "thresholds.csv"), index=False, encoding=ENC, float_format="%.17g")
references.to_csv(os.path.join(OUT_DIR, "train_reference.csv"), index=False, encoding=ENC, float_format="%.17g")
fit_rows.to_csv(os.path.join(OUT_DIR, "fit_rows.csv"), index=False, encoding=ENC)


### 6. 구역별 결과표

- 후보 수와 비율은 각 구역에서 판정한 행을 기준으로 계산함
- v3와 1분 CSV는 해당 구역의 라벨과 겹친 수를 남김
  - 라벨은 규칙으로 만든 후보이며 실제 고장 정답은 아님
- v1과 v2의 행 라벨은 참고용으로만 보존함
  - 행 라벨을 모든 구역의 정답으로 복사하지 않음
- `117`, `121`, `133`은 같은 CSV와 같은 구역에서 공통으로 판정한 행만 비교함
  - 원본 기록 행수와 1분 행수를 서로 같은 표본 수로 보지 않음

In [ ]:
# 결과를 다시 쓰기 전에 이전 실행의 완료 표시를 지움.
completion_file = os.path.join(OUT_DIR, "settings.json")
if os.path.isfile(completion_file):
    os.remove(completion_file)

method_names = {
    "fixed_9_2_3": "고정 9/2/3",
    "same_grade_history": "이전 동일 강종",
    "all_history": "이전 모든 강종",
    "first_lot_60_20_20": "001 시간 60/20/20",
    "within_lot_60_20_20": "같은 강종 없음: LOT 내부 60/20/20",
}
metric_rows = []
for (case, method, role, lot), block in predictions.groupby(["CASE", "METHOD", "ROLE", "LOT_NO"], sort=False):
    for group in groups:
        labels = block[group + "__LABEL"] if group + "__LABEL" in block else pd.Series(np.nan, index=block.index)
        row_labels = block["LABEL"] if "LABEL" in block else pd.Series(np.nan, index=block.index)
        flag = block[group + "__FLAG"]
        pending = block.loc[flag.isna(), group + "__STATUS"].value_counts()
        pending_text = []
        for status, count in pending.items():
            pending_text.append(str(status) + " " + str(count) + "행")

        known = labels.isin([0, 1]) & flag.notna()
        tp = int((known & labels.eq(1) & flag.eq(1)).sum())
        fp = int((known & labels.eq(0) & flag.eq(1)).sum())
        fn = int((known & labels.eq(1) & flag.eq(0)).sum())
        tn = int((known & labels.eq(0) & flag.eq(0)).sum())
        metric_rows.append({
            "MODEL": "IsolationForest",
            "N_ROWS": int(known.sum()),
            "N1": int((known & labels.eq(1)).sum()),
            "TP": tp, "FP": fp, "FN": fn, "TN": tn,
            "PRECISION_MEAN": tp / (tp + fp) if tp + fp else np.nan,
            "RECALL_MEAN": tp / (tp + fn) if tp + fn else np.nan,
            "F1_MEAN": 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else np.nan,
            "COVERAGE": float(known.mean()),
            "CASE": case,
            "METHOD": method,
            "ROLE": role,
            "LOT_NO": lot,
            "GRADE": block["GRADE"].iloc[0],
            "GROUP": group,
            "RAW_ROWS": len(block),
            "DATA_KIND": DATA_KIND,
            "SCORED_ROWS": int(flag.notna().sum()),
            "CANDIDATE_ROWS": int(flag.eq(1).sum()),
            "CANDIDATE_RATE": flag.mean(),
            "PENDING_ROWS": int(flag.isna().sum()),
            "PENDING_REASON": ", ".join(pending_text),
            "GROUP_LABEL_COLUMN": group + "__LABEL" if group + "__LABEL" in block else "",
            "GROUP_LABEL_KNOWN_IN_SCORED": int((labels.isin([0, 1]) & flag.notna()).sum()),
            "GROUP_LABEL1_IN_SCORED": int((labels.eq(1) & flag.notna()).sum()),
            "CANDIDATE_AND_GROUP_LABEL1": int((labels.eq(1) & flag.eq(1)).sum()),
            "ROW_LABEL1_IN_SCORED": int((row_labels.eq(1) & flag.notna()).sum()) if "LABEL" in block else np.nan,
            "CANDIDATE_AND_ROW_LABEL1": int((row_labels.eq(1) & flag.eq(1)).sum()) if "LABEL" in block else np.nan,
        })

metrics = pd.DataFrame(metric_rows)
metrics.to_csv(os.path.join(OUT_DIR, "metrics.csv"), index=False, encoding=ENC, float_format="%.7g")
show_names = {
    "METHOD": "학습 방법",
    "ROLE": "확인 구간",
    "LOT_NO": "LOT",
    "GROUP": "함께 본 구역",
    "RAW_ROWS": "입력 행",
    "GROUP_LABEL_KNOWN_IN_SCORED": "구역 라벨이 있는 판정 행",
    "GROUP_LABEL1_IN_SCORED": "판정한 행의 구역 라벨 1",
    "CANDIDATE_AND_GROUP_LABEL1": "구역 후보와 구역 라벨 1이 겹친 수",
    "SCORED_ROWS": "판정한 행",
    "CANDIDATE_ROWS": "구역 후보 행",
    "CANDIDATE_RATE": "구역 후보 비율",
    "PENDING_ROWS": "판정 보류 행",
    "PENDING_REASON": "보류 이유",
    "ROW_LABEL1_IN_SCORED": "판정한 행의 기존 라벨 1",
    "CANDIDATE_AND_ROW_LABEL1": "구역 후보와 기존 행 라벨 1이 겹친 수",
}
show_names.update({
    "N_ROWS": "비교 행", "TP": "찾은 구역 1", "FP": "기존 0을 1로 예측",
    "FN": "놓친 구역 1", "TN": "기존 0을 0으로 예측",
    "PRECISION_MEAN": "Precision", "RECALL_MEAN": "Recall", "F1_MEAN": "F1",
})
show_columns = [
    "METHOD", "ROLE", "LOT_NO", "GROUP", "SCORED_ROWS",
    "CANDIDATE_ROWS", "CANDIDATE_RATE", "PENDING_ROWS", "PENDING_REASON",
]
if LABEL_COLUMNS:
    show_columns = show_columns + ["GROUP_LABEL1_IN_SCORED", "TP", "TN", "FP", "FN", "PRECISION_MEAN", "RECALL_MEAN", "F1_MEAN"]
print("각 구역의 판정 결과. 후보 비율의 분모는 그 구역에서 판정한 행임.")
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(metrics[show_columns].replace({"METHOD": method_names, "GROUP": group_titles}).rename(columns=show_names).round(4))

# 같은 LOT, 같은 구역에서 계산 가능한 방법들의 공통 행만 비교한다.
comparison_rows = []
for lot in TEST_LOTS:
    lot_predictions = predictions.loc[predictions["LOT_NO"].eq(lot) & predictions["ROLE"].eq("test") & predictions["METHOD"].isin(["fixed_9_2_3", "same_grade_history", "all_history"])]
    cases = list(lot_predictions.groupby(["CASE", "METHOD"], sort=False))
    for group in groups:
        common_ids = None
        available_methods = []
        for (case, method), block in cases:
            ids = set(block.loc[block[group + "__FLAG"].notna(), "ROW_ID"])
            if ids:
                available_methods.append(method)
                common_ids = ids if common_ids is None else common_ids.intersection(ids)
        if common_ids is None:
            common_ids = set()

        for (case, method), block in cases:
            flag = block[group + "__FLAG"]
            common = block.loc[block["ROW_ID"].isin(common_ids) & flag.notna()]
            common_flag = common[group + "__FLAG"]
            if not flag.notna().any():
                status = ", ".join(block[group + "__STATUS"].dropna().unique())
            elif len(available_methods) < 2:
                status = "비교 가능한 방법이 1개뿐임"
            elif not len(common):
                status = "방법 사이에 공통으로 판정한 행 없음"
            else:
                status = "같은 행에서 비교"
            comparison_rows.append({
                "CASE": case,
                "METHOD": method,
                "LOT_NO": lot,
                "GROUP": group,
                "COMMON_METHODS": ", ".join(available_methods),
                "N_AVAILABLE_METHODS": len(available_methods),
                "COMMON_ROWS": len(common),
                "CANDIDATE_ROWS": int(common_flag.eq(1).sum()),
                "CANDIDATE_RATE": common_flag.mean(),
                "STATUS": status,
            })

comparison = pd.DataFrame(comparison_rows)
comparison.to_csv(os.path.join(OUT_DIR, "comparison.csv"), index=False, encoding=ENC, float_format="%.7g")
comparison_names = dict(show_names)
comparison_names.update({
    "COMMON_ROWS": "방법 간 공통 판정 행",
    "STATUS": "비교 상태",
})
print("117, 121, 133: 구역별로 같은 행에서 방법 비교")
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(comparison[["LOT_NO", "GROUP", "METHOD", "COMMON_ROWS", "CANDIDATE_ROWS", "CANDIDATE_RATE", "STATUS"]].replace({"METHOD": method_names, "GROUP": group_titles}).rename(columns=comparison_names).round(4))


### 7. 모든 LOT와 구역을 Figure 하나에서 비교

- 같은 위치의 칸은 같은 LOT와 구역임.
  - F1, 잘못 잡은 0, 놓친 1, 평가한 비율을 함께 봄.
- `보류`는 평가하지 못한 경우이며 정상 0이나 성능 0이 아님.
  - 흰색 `—`는 해당 실험의 평가 대상이 아님.
- LOT 내부 분할 그림에는 뒤 20% Test를 표시함.
  - Valid는 위 결과표에서 확인함.

In [ ]:
# 모든 구역을 같은 순서로 놓고, 한 실행의 결과를 Figure 하나에 모음.
# 원래 라벨과 모델 예측은 바꾸지 않고 저장한 지표만 그림으로 표시함.
plot_methods = ["fixed_9_2_3", "same_grade_history", "all_history"]
if metrics["METHOD"].eq("first_lot_60_20_20").any():
    plot_methods.append("first_lot_60_20_20")

if metrics["METHOD"].eq("within_lot_60_20_20").any():
    plot_methods.append("within_lot_60_20_20")

plot_groups = list(groups)
plot_lots = LOTS
panels = [
    ("IsolationForest", "F1_MEAN", "기존 구역 라벨과의 F1", "Blues"),
    ("IsolationForest", "FP", "기존 0을 1로 예측한 수", "Oranges"),
    ("IsolationForest", "FN", "기존 1을 0으로 예측한 수", "Oranges"),
    ("IsolationForest", "COVERAGE", "전체 기록 중 평가한 비율", "Greens"),
]

fig, axes = plt.subplots(
    len(plot_methods), len(panels),
    figsize=(25, 6.1 * len(plot_methods)),
    squeeze=False,
)

# 횟수 그림은 학습 방법이 달라도 같은 색 범위를 사용함.
count_limits = {}
for column in ["FP", "FN"]:
    count_limits[column] = max(1, metrics.loc[
        metrics["LOT_NO"].ne("전체") & metrics["MODEL"].eq("IsolationForest"), column
    ].max())

for method_index, method in enumerate(plot_methods):
    for panel_index, (model, column, title, color_map) in enumerate(panels):
        axis = axes[method_index, panel_index]
        rows = metrics.loc[
            metrics["METHOD"].eq(method) & metrics["MODEL"].eq(model)
            & metrics["LOT_NO"].ne("전체")
        ].copy()
        if method in ["first_lot_60_20_20", "within_lot_60_20_20"]:
            rows = rows.loc[rows["ROLE"].eq("test")].copy()
        rows["LOT_NO"] = rows["LOT_NO"].astype(str)
        values = rows.pivot(index="LOT_NO", columns="GROUP", values=column)
        values = values.reindex(index=plot_lots, columns=plot_groups)
        compared = rows.pivot(index="LOT_NO", columns="GROUP", values="N_ROWS")
        compared = compared.reindex(index=plot_lots, columns=plot_groups)
        exists = rows.pivot(index="LOT_NO", columns="GROUP", values="RAW_ROWS")
        exists = exists.reindex(index=plot_lots, columns=plot_groups)

        # 비교하지 못한 칸은 성능 0이나 오류 0으로 표시하지 않음.
        # 평가 비율만 0% 그대로 보여, 평가하지 못했다는 사실을 남김.
        if column != "COVERAGE":
            values = values.where(compared.gt(0))
        if column == "F1_MEAN":
            denominator = rows["TP"] * 2 + rows["FP"] + rows["FN"]
            rows["F1_DEFINED"] = denominator.gt(0)
            defined = rows.pivot(index="LOT_NO", columns="GROUP", values="F1_DEFINED")
            values = values.where(defined.reindex(index=plot_lots, columns=plot_groups).eq(True))

        colors = plt.get_cmap(color_map).copy()
        colors.set_bad("#e5e7eb")
        maximum = count_limits[column] if column in count_limits else 1
        heat = axis.imshow(values.to_numpy(dtype=float), cmap=colors, vmin=0, vmax=maximum, aspect="auto")

        for lot_index, lot in enumerate(plot_lots):
            for group_index, group in enumerate(plot_groups):
                value = values.loc[lot, group]
                if pd.isna(exists.loc[lot, group]):
                    text = "—"
                    axis.add_patch(plt.Rectangle(
                        (group_index - 0.5, lot_index - 0.5), 1, 1,
                        facecolor="white", edgecolor="none",
                    ))
                elif column != "COVERAGE" and compared.loc[lot, group] == 0:
                    text = "보류"
                elif pd.isna(value):
                    text = "정의\n불가"
                elif column == "COVERAGE":
                    text = str(round(value * 100)) + "%"
                elif column == "F1_MEAN":
                    text = format(value, ".2f")
                else:
                    text = str(int(round(value)))

                text_color = "white" if pd.notna(value) and value > maximum * 0.65 else "#243447"
                axis.text(group_index, lot_index, text, ha="center", va="center", fontsize=7.5, color=text_color)

        axis.set_xticks(range(len(plot_groups)), plot_groups, rotation=55, ha="right", fontsize=9)
        axis.set_yticks(range(len(plot_lots)), [lot[-3:] for lot in plot_lots], fontsize=9)
        axis.set_title(method_names[method] + "\n" + title, fontsize=13, pad=12)
        axis.set_ylabel("LOT 번호 끝 3자리")
        axis.set_xticks(np.arange(len(plot_groups) + 1) - 0.5, minor=True)
        axis.set_yticks(np.arange(len(plot_lots) + 1) - 0.5, minor=True)
        axis.grid(which="minor", color="white", linewidth=0.7)
        axis.tick_params(which="minor", bottom=False, left=False)
        fig.colorbar(heat, ax=axis, fraction=0.035, pad=0.02)

fig.suptitle(DATA_FILE + " | 모든 LOT와 구역의 기존 라벨, 모델 예측 비교", fontsize=22, y=0.985)
fig.text(
    0.035, 0.945,
    "각 칸은 같은 LOT와 같은 구역임. F1은 높을수록 기존 라벨과 잘 맞음. 주황색 수는 적을수록 좋음.\n"
    "흰색 —: 해당 실험의 평가 대상 아님. 회색 보류: 평가하지 못함. 정의 불가: 실제 1과 예측 1이 모두 없어 F1을 계산할 수 없음.",
    fontsize=12, va="top",
)
fig.text(
    0.035, 0.018,
    "고정 9/2/3: 108, 114는 Valid / 117, 121, 133은 Test. 이전 강종 방법: 각 LOT보다 앞선 자료로 학습함. LOT 내부 분할은 뒤 20% Test만 그림에 표시함.\n"
    "각 구역의 기존 라벨을 기준으로 비교함. 학습 방법끼리의 공통 Test 행 비교는 위 comparison 표를 봄.\n"
    "기존 라벨 1은 이상 후보이며 고장 정답이 아님. 라벨은 학습이나 후보 기준 계산에 넣지 않음.",
    fontsize=11,
)
fig.subplots_adjust(left=0.035, right=0.985, top=0.885, bottom=0.095, hspace=0.42, wspace=0.32)
figure_path = os.path.join(FIG_DIR, "if_all_lots_all_groups.png")
fig.savefig(figure_path, dpi=150, facecolor="white")
plt.show()
plt.close(fig)
print("그림 한 장:", figure_path)


### 8. 저장한 결과

- `predictions.csv`는 시각, 작업 기록, 라벨, 구역별 점수와 판정 이유임
- `metrics.csv`와 `comparison.csv`는 구역 결과와 같은 행에서 비교한 결과임
- `thresholds.csv`, `train_reference.csv`, `fit_rows.csv`는 Train 기준과 학습 표본임
- `operating_context.csv`, `split_plan.csv`, `settings.json`은 작업 조건, 분할, 실행 설정임
  - CSV와 조업 원본 3개의 내용 식별값도 저장함
- 같은 강종이나 작업 조건의 학습이 없으면 보류하며 정상 0으로 바꾸지 않음
- `Z1~Z8`의 구역 후보는 같은 번호의 TC와 OP를 함께 본 결과임
  - `CLN`과 `Z9`는 각각 한 센서만 사용함
  - `COOLING`은 `TC-OUT1`과 `TC-OUT2`를 함께 사용함
  - 원인, 실제 강판 온도, 품질 문제를 확정한 결과는 아님

- 선택한 CSV와 생성 정보의 내용이 맞을 때만 생성 근거를 연결함
  - 맞지 않으면 이유를 표시하고 CSV 안의 값으로 모델을 실행함

In [ ]:
settings = {
    "model_version": "joint_groups_conditioned_v34_compatible",
    "data_kind": DATA_KIND,
    "dataset_version": DATASET_VERSION,
    "source_manifest_status": manifest_status,
    "source_manifest": source_manifest,
    "run_started_at": RUN_ID.rsplit("__", 1)[-1],
    "run_completed_at": datetime.now().isoformat(),
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "input_file": DATA_FILE,
    "input_sha256": input_hashes,
    "run_id": RUN_ID,
    "sklearn_version": sklearn.__version__,
    "seed": SEED,
    "n_estimators": N_ESTIMATORS,
    "max_fit_rows_per_group_context": MAX_FIT_ROWS,
    "max_samples": MAX_SAMPLES,
    "train_quantile": TRAIN_QUANTILE,
    "groups": groups,
    "grade_groups": GRADE_GROUPS,
    "normalization": "Train context/grade median and IQR; zero IQR falls back to Train std then 1",
    "context_models": "separate model for each maintenance/repair context",
    "grade_change_input": "log1p(minutes since past actual CR1 grade-change event), grade-sensitive groups only",
    "reference_min_rows": MIN_REFERENCE_ROWS,
    "reference_min_observed_minutes": MIN_REFERENCE_MINUTES,
    "threshold": "Train same context/grade score quantile, strict greater-than",
    "no_row_level_prediction": True,
    "label_use": "group candidate-label overlap for v3/v4; historical row-label overlap only for v1/v2; no labels in fitting or threshold calculation",
    "label_columns": LABEL_COLUMNS,
    "group_input_columns": {group: values.columns.tolist() for group, values in group_inputs.items()},
    "feature_source": "precomputed raw60s group means in minute CSV" if DATA_KIND == "minute" else "causal raw sensor features calculated in this notebook",
    "context_source": "preserved from minute CSV; current raw context file hashes are recorded without rejoining" if DATA_KIND == "minute" else "raw context files joined at each observation",
    "reference_min_rows_raw": RAW_MIN_REFERENCE_ROWS,
    "reference_min_rows_minute": MIN_MINUTE_REFERENCE_ROWS,
    "reference_unit": "valid minute rows" if DATA_KIND == "minute" else "valid raw observation rows",
    "rolling_seconds": 60,
    "rolling_min_past_rows": MIN_PAST_ROWS,
    "rolling_closed": "left",
    "rolling_std_ddof": 1,
    "gap_reset_seconds": MAX_GAP_SECONDS,
    "validity": "group-specific finite inputs, excludes existing EXCLUDED, unmatched grade events withheld",
    "fit_sampling": "same quota per available Train LOT, short LOT uses all available rows",
    "fixed_roles": {"train": TRAIN_LOTS, "valid": VALID_LOTS, "test": TEST_LOTS},
    "run_first_lot": RUN_FIRST_LOT,
    "run_no_history_lot": RUN_NO_HISTORY_LOT,
    "first_lot_policy": "OBS_START min to OBS_END max defines cuts; AVAILABLE_AT must precede train/valid end; boundary-straddling minutes withheld",
    "plot_groups": list(groups),
    "plot_layout": "one figure, all LOTs and groups, per-method metrics",
    "plot_aggregation": "same LOT and group candidate-label agreement, with withheld coverage",
    "minute_no_rolling_recalculation": DATA_KIND == "minute",
    "operating_context_limit": "work intervals and team exclusions are retrospective; no explicit running/stopped command logs",
}
with open(os.path.join(OUT_DIR, "settings.json"), "w", encoding="utf-8") as settings_file:
    json.dump(settings, settings_file, ensure_ascii=False, indent=2)
print("저장 완료:", OUT_DIR)
print("그림:", FIG_DIR)
